In [4]:
!pip install -Uq unstructured[all-docs]
!pip install -Uq sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 752.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.


In [5]:
import json
import base64
import torch
import faiss
import numpy as np
from typing import List

# Unstructured
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# Hugging Face
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


In [6]:
# ---- Embedding model ----
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ---- LLM (free, public) ----
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
def partition_document(file_path: str):
    print(f"📄 Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )

    print(f"✅ Extracted {len(elements)} elements")
    return elements


In [8]:
def create_chunks_by_title(elements):
    print("🔨 Creating smart chunks...")

    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )

    print(f"✅ Created {len(chunks)} chunks")
    return chunks


In [9]:
def separate_content_types(chunk):
    content = {
        "text": chunk.text,
        "tables": [],
        "images": []
    }

    if hasattr(chunk.metadata, "orig_elements"):
        for el in chunk.metadata.orig_elements:
            if el.category == "Table":
                content["tables"].append(
                    getattr(el.metadata, "text_as_html", el.text)
                )
            elif el.category == "Image":
                if hasattr(el.metadata, "image_base64"):
                    content["images"].append(el.metadata.image_base64)

    return content


In [10]:
def generate_summary(text, tables):
    prompt = f"""
Summarize the following document content for retrieval.
Include key facts, concepts, and questions it can answer.

TEXT:
{text}

TABLES:
{tables}

SUMMARY:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    output = llm.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.2
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [11]:
def summarise_chunks(chunks):
    processed = []

    for i, chunk in enumerate(chunks):
        print(f"🧠 Processing chunk {i+1}/{len(chunks)}")

        data = separate_content_types(chunk)

        summary = generate_summary(
            data["text"],
            data["tables"]
        )

        processed.append({
            "content": summary,
            "metadata": {
                "raw_text": data["text"],
                "tables": data["tables"],
                "images_base64": data["images"]
            }
        })

    return processed


In [12]:
def create_vector_store(chunks):
    texts = [c["content"] for c in chunks]
    embeddings = embedding_model.encode(texts)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return index, chunks


In [13]:
def retrieve(query, index, chunks, k=3):
    query_emb = embedding_model.encode([query])
    _, ids = index.search(np.array(query_emb), k)

    return [chunks[i] for i in ids[0]]


In [14]:
def generate_final_answer(retrieved_chunks, query):
    context = ""

    for i, chunk in enumerate(retrieved_chunks):
        context += f"\n--- Chunk {i+1} ---\n"
        context += chunk["metadata"]["raw_text"]

    prompt = f"""
Answer the question using ONLY the context below.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    output = llm.generate(**inputs, max_new_tokens=256)

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [20]:
def run_pipeline(pdf_path, query):
    elements = partition_document(pdf_path)
    chunks = create_chunks_by_title(elements)
    processed = summarise_chunks(chunks)
    index, stored_chunks = create_vector_store(processed)
    retrieved = retrieve(query, index, stored_chunks)
    return generate_final_answer(retrieved, query)


In [19]:
answer = run_pipeline(
    "/content/paper.pdf",
    "What are the main components of the Transformer?"
)

print(answer)


📄 Partitioning document: /content/paper.pdf


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

...

✅ Extracted 173 elements
🔨 Creating smart chunks...
✅ Created 21 chunks
🧠 Processing chunk 1/21
🧠 Processing chunk 2/21
🧠 Processing chunk 3/21
🧠 Processing chunk 4/21
🧠 Processing chunk 5/21
🧠 Processing chunk 6/21
🧠 Processing chunk 7/21
🧠 Processing chunk 8/21
🧠 Processing chunk 9/21
🧠 Processing chunk 10/21
🧠 Processing chunk 11/21
🧠 Processing chunk 12/21
🧠 Processing chunk 13/21
🧠 Processing chunk 14/21
🧠 Processing chunk 15/21
🧠 Processing chunk 16/21


Token indices sequence length is longer than the specified maximum sequence length for this model (2545 > 2048). Running this sequence through the model will result in indexing errors


🧠 Processing chunk 17/21


This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


🧠 Processing chunk 18/21
🧠 Processing chunk 19/21
🧠 Processing chunk 20/21
🧠 Processing chunk 21/21

Answer the question using ONLY the context below.

CONTEXT:

--- Chunk 1 ---
6.2 Model Variations

To evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the development set, newstest2013. We used beam search as described in the previous section, but no checkpoint averaging. We present these results in Table 3.

In Table 3 rows (A), we vary the number of attention heads and the attention key and value dimensions, keeping the amount of computation constant, as described in Section 3.2.2. While single-head attention is 0.9 BLEU worse than the best setting, quality also drops off with too many heads.

5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively.

8

Table 3: Variations on the Transformer architecture. Unlisted value

In [18]:
!apt-get update && apt-get install -y poppler-utils


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,225 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Pac